In [ ]:
import geopandas as gpd
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm
import numpy as np
import seaborn as sns

import scipy.stats as stats

from cafo_iowa.estimate.estimate import load_and_process_facilities
from cafo_iowa.utils.utils import save_geojson_with_lists

In [ ]:
facilities = load_and_process_facilities(
    match_barns_to_empty_facilities=True, buffer_size=500
)

# save data for manual inspection
save_geojson_with_lists(facilities, "output/data/facilities.geojson")

In [ ]:
facilities.head()

In [ ]:
# select all facilities that have swine
facilities_swine = facilities[facilities.reported_swine_animal_units > 0].copy()

# Ensure no missing values in the category
facilities_swine["swine_cat"] = facilities_swine["swine_cat"].fillna("None")

In [ ]:
print("Facilities: ", len(facilities))
print("Barns in facilities: ", facilities["barn_ids"].explode().nunique())
print("Facilities with swine: ", len(facilities_swine))
print(
    "Barns in facilities with swine: ",
    facilities_swine["barn_ids"].explode().nunique(),
)
print("Facilities with no barns: ", len(facilities[facilities["barn_ids"].isna()]))
print(
    "Swine facilities with no barns: ",
    len(facilities_swine[facilities_swine["barn_ids"].isna()]),
)
print("Mean barns per facility: ", round(facilities["barn_count"].mean(), 2))
print(
    "Mean barns per swine facility: ", round(facilities_swine["barn_count"].mean(), 2)
)

In [ ]:
# histogram of total barn area
fig, ax = plt.subplots()
sns.histplot(facilities_swine["barn_area_sqm"], bins=100, ax=ax)
ax.set_xlabel("Barn area (sqm)")
ax.set_ylabel("Count")
ax.set_title("Histogram of barn area across swine facilities")
plt.savefig("output/plots/barn_area_hist.png", dpi=300)
plt.show()

In [ ]:
def plot_swine_facilities_loop(
    data,
    category_col="swine_cat_combined",
    x_col="reported_animal_units",
    y_col="barn_area_sqm",
    color_col="permit_count",
    n_cols=4,
    base_figsize=5,
    palette="Set2",
    alpha=0.8,
    reference_lines=(500, 1000, 2000),
    log_scale=True,
    add_xy_line=False,
    save=False,
    save_path=None,
):
    # 1) Identify categories
    categories = data[category_col].unique()
    n_cats = len(categories)

    # 2) Determine the grid of subplots
    n_rows = int(np.ceil(n_cats / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(base_figsize * n_cols, base_figsize * n_rows),
        sharex=True,
        sharey=True,
    )

    # 3) Flatten axes for easy iteration
    if isinstance(axes, np.ndarray):
        axes = axes.flatten()
    else:
        axes = [axes]

    # 4) Prepare color mapping
    unique_colors = sorted(data[color_col].dropna().unique())
    palette_values = sns.color_palette(palette, len(unique_colors))
    color_map = dict(zip(unique_colors, palette_values))

    # 5) Loop through each category and plot
    for i, category in enumerate(categories):
        ax = axes[i]
        subset = data[data[category_col] == category]

        # Scatter plot per color group
        for color_val in unique_colors:
            sub2 = subset[subset[color_col] == color_val]
            ax.scatter(
                sub2[x_col],
                sub2[y_col],
                color=color_map[color_val],
                label=color_val,
                alpha=alpha,  # Applied here
            )
            # Add x=y line if specified
            if add_xy_line:
                ax.plot(
                    sub2[x_col].unique(),
                    sub2[x_col].unique(),
                    linestyle="--",
                    color="black",
                )

        # Set log scale if requested
        if log_scale:
            ax.set_xscale("log")
            ax.set_yscale("log")

        # Add vertical reference lines
        if reference_lines:
            for ref_line in reference_lines:
                ax.axvline(x=ref_line, linestyle="--", color="gray", alpha=0.7)

        # Compute correlation (Pearson)
        sub_corr = subset[[x_col, y_col]].dropna()
        if len(sub_corr) > 1:
            r = sub_corr[x_col].corr(sub_corr[y_col])
            corr_str = f"r={r:.2f}"
        else:
            corr_str = "r=NA"

        # Annotate correlation in top-left corner
        ax.text(
            0.05,
            0.95,
            corr_str,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=10,
            bbox=dict(boxstyle="round", fc="white", alpha=0.7, edgecolor="none"),
            fontweight="bold",
        )

        # Title with sample size
        n = len(subset)
        ax.set_title(f"{category} (n={n})")

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    # Build a figure-level legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, title=color_col, loc="center left", bbox_to_anchor=(0.91, 0.5)
    )

    # Shared axis labels
    scale_label = " (log scale)" if log_scale else ""
    fig.text(0.5, 0.02, f"{x_col}{scale_label}", ha="center")
    fig.text(
        0.02,
        0.5,
        f"{y_col}{scale_label}",
        va="center",
        rotation="vertical",
    )

    # Adjust layout with smaller padding
    plt.tight_layout(rect=[0.02, 0.02, 0.88, 0.96])
    plt.subplots_adjust(
        left=0.08, bottom=0.08, right=0.85, top=0.92, wspace=0.3, hspace=0.3
    )

    # Saving logic
    if save:
        if save_path is None:
            raise ValueError("save_path must be provided if save=True")
        plt.savefig(
            save_path, dpi=300, bbox_inches="tight"
        )  # Ensures everything is in view

    plt.show()

In [ ]:
plot_swine_facilities_loop(
    facilities_swine,
    category_col="swine_cat_combined",
    x_col="reported_animal_units",
    y_col="barn_area_sqm",
    color_col="permit_count",
    save=True,
    save_path="output/plots/swine_facilities.png",
)

In [ ]:
plot_swine_facilities_loop(
    facilities_swine,
    category_col="swine_cat_combined",
    x_col="reported_animal_units",
    y_col="estimated_animal_units",
    color_col="merged_nearby_barns",
    add_xy_line=True,
    log_scale=True,
    save=True,
    save_path="output/plots/reported_vs_estimated_aus.png",
)

In [ ]:
# filter facilities to only up to 5000 reported animal units, and remove swine_and_other_animals category
df = facilities_swine[
    (facilities_swine["reported_animal_units"] <= 2500)
    & (facilities_swine["estimated_animal_units"] <= 2500)
    & (facilities_swine["swine_cat_combined"].isin(["swine_wean", "swine_grow"]))
].copy()

plot_swine_facilities_loop(
    df,
    category_col="swine_cat_combined",
    x_col="reported_animal_units",
    y_col="estimated_animal_units",
    color_col="merged_nearby_barns",
    alpha=0.5,
    add_xy_line=True,
    log_scale=False,
    save=True,
    save_path="output/plots/reported_vs_estimated_aus.png",
)

In [ ]:
# create a histogram of the reported and estimated animal units
fig, ax = plt.subplots()
ax.set_xlim(0, 7000)
ax.axvline(x=500, linestyle="--", color="black")
ax.axvline(x=1000, linestyle="--", color="black")
ax.axvline(x=2000, linestyle="--", color="black")
sns.histplot(
    facilities_swine["reported_animal_units"],
    bins=200,
    ax=ax,
    color="blue",
    label="Reported",
    alpha=0.5,
)
sns.histplot(
    facilities_swine["estimated_animal_units"],
    bins=200,
    ax=ax,
    color="red",
    label="Estimated",
    alpha=0.3,
)
ax.set_xlabel("Animal units")
# add vertical lines at 500, 1000, 2000
ax.set_ylabel("Count")
ax.set_title("Histogram of reported and estimated animal units")
plt.legend()
plt.savefig("output/plots/aus_hist.png", dpi=300)
plt.show()

In [ ]:
bin_edges = np.linspace(0, 4000, 101)  # 101 edges = 100 bins

# Create the figure and axes for two side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Data to plot and titles
data_to_plot = [
    (facilities_swine["reported_animal_units"], "Facility Animal Units"),
    (facilities_swine["permit_animal_units"].explode(), "Permit Animal Units"),
]

# Colors for the histograms
colors = ["skyblue", "lightcoral"]

# Loop through axes and datasets
for ax, (data, title), color in zip(axes, data_to_plot, colors):
    sns.histplot(data, bins=bin_edges, ax=ax, color=color)
    ax.axvline(x=500, color="red", linestyle="--", label="500")
    ax.axvline(x=1000, color="blue", linestyle="--", label="1000")
    ax.axvline(x=2000, color="green", linestyle="--", label="2000")
    ax.set_xlim(0, 4000)
    ax.set_xlabel(title)
    ax.legend(title="Thresholds")
    ax.set_title(f"Histogram of {title}")

# Set shared y-axis label
axes[0].set_ylabel("Count")

# Adjust layout
plt.tight_layout()

# Save the figure
plt.savefig("output/plots/animal_units_vs_raw_animal_units.png", dpi=300)

# Show the plots
plt.show()

In [ ]:
# subset swine facilities to group of interest
df = facilities_swine

In [ ]:
# Filter the two groups
group1 = df[
    (df["reported_animal_units"] >= 900) & (df["reported_animal_units"] <= 1000)
]["estimated_animal_units"]
group2 = df[
    (df["reported_animal_units"] >= 1001) & (df["reported_animal_units"] <= 1100)
]["estimated_animal_units"]

# Print number of observations in each group
n1, n2 = len(group1), len(group2)
print(f"Number of observations in Group 1 (900-1000 animal units): {n1}")
print(f"Number of observations in Group 2 (1001-1100 animal units): {n2}")

# Levene's test (for variance equality, robust to non-normality)
levene_stat, levene_p = stats.levene(group1, group2)

# Bartlett’s test (for variance equality, assumes normality)
bartlett_stat, bartlett_p = stats.bartlett(group1, group2)

print(f"\nLevene's Test: Statistic={levene_stat:.4f}, p-value={levene_p:.4f}")
print(f"Bartlett's Test: Statistic={bartlett_stat:.4f}, p-value={bartlett_p:.4f}")

# Interpretation
if levene_p < 0.05:
    print("Levene's test suggests variances are significantly different.")
else:
    print("Levene's test suggests variances are not significantly different.")

if bartlett_p < 0.05:
    print("Bartlett's test suggests variances are significantly different.")
else:
    print("Bartlett's test suggests variances are not significantly different.")